# Deploying and Evaluating the Agent

Notebook 01 left you with a working agent and no way to give it to anyone. It
lives in a notebook, it holds an open Neo4j driver, and it reads your
credentials from `dbutils`. None of that survives being handed to a colleague.

Here you take the same graph, log it as an MLflow model, deploy it to a Model
Serving endpoint, and score it against a question set with MLflow's judges.

**Prerequisites**

| Lab | What this notebook needs from it |
|---|---|
| [Notebook 01](01_langgraph_agent.ipynb) | A run that reached the end. Everything here uses the same three tools |
| [Lab 3 notebook 01](../Lab_3_Semantic_Search/01_data_and_embeddings.ipynb) | The `fleet-ops-<your-user>` secret scope |
| [Lab 4 Part A](../Lab_4_Compound_AI_Agents/PART_A.md) | Your Genie space, and its space ID |

**Learning objectives**

- Read the three assumptions a notebook agent makes that a served one cannot
- Declare the resources an endpoint is allowed to reach, and pass credentials it is not
- Log a models-from-code agent to Unity Catalog and deploy it
- Query the endpoint and confirm all three tools still answer
- Score routing with a deterministic scorer and answers with an LLM judge


## What changes when the agent leaves the notebook

Three assumptions in notebook 01 hold only inside a notebook.

**`dbutils` exists.** Notebook 01 reads your Aura password out of the secret
scope with `dbutils.secrets.get`. A serving container has no `dbutils` and no
notebook user. It gets its credentials as environment variables, bound to
secret references, resolved by the serving control plane when the endpoint
starts.

**You are the one asking.** In the notebook, every Databricks call runs as you.
The endpoint runs as a service principal that Model Serving creates, and that
principal starts with access to nothing. What it may reach is declared at log
time, as a list of resources, and granted automatically from that list.

**The wiring is a cell you already ran.** A model has to build itself from
nothing on a machine you will never see. So the wiring moves into `agent.py`,
which is a file rather than a notebook, and MLflow loads that file as the model.

`agent.py` sits beside `tools.py` in this folder. Open it. It imports the same
node builders notebook 01 used and wires the same graph; what is new is
`build_runtime`, which opens the connections from environment variables, and
`FleetOpsAgent`, which is the MLflow interface Model Serving speaks.


## Section 1: Configuration

The same Genie space ID as notebook 01, and the same secret scope, derived from
`current_user()` so there is nothing to copy across.

The model name and the endpoint name come from `agent.py` rather than from a
string you type. Lab 6 redeploys **this** endpoint with memory added rather than
standing up a second one, so the name is a contract between the two labs.


In [1]:
# ==================================================
# CONFIGURATION - replace GENIE_SPACE_ID with yours
# ==================================================

# From Lab 4 Part A. Open your Genie space and take the ID out of the URL:
#   https://<workspace>/genie/rooms/<GENIE_SPACE_ID>
GENIE_SPACE_ID = "01f1661b55731a0293c3f84ac9c5ba52"

import sys

sys.path.insert(0, ".")

from agent import UC_MODEL_NAME, endpoint_name
from tools import secret_scope_name

CURRENT_USER = spark.sql("SELECT current_user()").collect()[0][0]
SECRET_SCOPE = secret_scope_name(spark)
ENDPOINT_NAME = endpoint_name(SECRET_SCOPE)

print(f"Secret scope:   {SECRET_SCOPE}")
print(f"Genie space ID: {GENIE_SPACE_ID}")
print(f"UC model:       {UC_MODEL_NAME}")
print(f"Endpoint:       {ENDPOINT_NAME}")


Secret scope:   fleet-ops-ryan-knight-neo4j-com
Genie space ID: 01f1661b55731a0293c3f84ac9c5ba52
UC model:       databricks-neo4j-workshop.agents.fleet_ops_assistant
Endpoint:       fleet-ops-assistant-ryan-knight-neo4j-com


## Section 2: Run the agent before you log it

A deploy takes about fifteen minutes and tells you almost nothing when it
fails. Running `agent.py` here first costs a minute and fails with a stack
trace, so do that.

`export_neo4j_env` reads the values out of your secret scope and writes them
into this process's environment under the same names the endpoint will use. The password is read, written, and dropped inside that call, so no cell
here binds it to a name.

`build_runtime` is then the whole of what the serving container does at startup:
open the driver, ask Aura which database it holds, build the three nodes, drop
`graphrag_node` if the vector index is missing, and compile the graph.


In [2]:
from agent import DEFAULT_CONFIG, build_runtime, export_neo4j_env

print("Environment variables set:", export_neo4j_env(dbutils, SECRET_SCOPE))

MODEL_CONFIG = dict(DEFAULT_CONFIG)
MODEL_CONFIG["genie_space_id"] = GENIE_SPACE_ID

runtime = build_runtime(MODEL_CONFIG)
print("Tools available to the supervisor:", runtime.available_tools)

state = runtime.graph.invoke(
    {
        "question": "What is the procedure for an EGT exceedance?",
        "trace": [],
        "findings": [],
    }
)
print("Route:", state["trace"])
print(state["answer"][:600])

runtime.driver.close()


Environment variables set: ('NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD', 'NEO4J_DATABASE')


Tools available to the supervisor: ('genie_node', 'cypher_node', 'graphrag_node')


Route: ['graphrag_node']
The procedure for an EGT exceedance is outlined as follows: "Reduce thrust to idle if flight conditions permit", then "Monitor EGT trend for stabilization", and if "EGT exceeds 695°C", "initiate engine shutdown procedures per QRH". After landing, the troubleshooting procedure involves reviewing "DFDR/QAR data for EGT exceedance duration", performing a "visual inspection of the exhaust nozzle", conducting a "borescope inspection of the HPT blades", verifying "EGT probe calibration", and checking the "fuel nozzle spray pattern". The severity of the exceedance is classified as "CRITICAL" if "EGT 


## Section 3: Resources, or what the endpoint is allowed to reach

The serving principal starts with access to nothing. Every Databricks thing the
agent touches has to be named at log time, and MLflow grants the principal
access to exactly that list.

Four kinds of thing, twelve resources:

- **the Genie space**, which `genie_node` asks
- **the SQL warehouse behind that space**, which is where the SQL Genie writes
  actually runs
- **the eight gold tables** that SQL reads, in `databricks-neo4j-workshop.aircraft`
- **`databricks-claude-sonnet-5`**, the supervisor and the two nodes
  that generate text
- **`databricks-bge-large-en`**, which embeds the question `graphrag_node` looks
  up

The list lives in `build_resources` in `agent.py` rather than in this cell,
because Lab 6 redeploys this endpoint and has to declare exactly the same thing.

The warehouse is the one that is easy to leave out, and leaving it out fails in
a way that looks like something else. Declaring the Genie space alone produces
an endpoint that deploys cleanly, routes correctly, and then answers every
sensor question with `is not authorized to use or monitor this SQL Endpoint`.
The space grants the space. The SQL underneath it is a separate resource with a
separate grant.

Your Aura instance is not on the list, and cannot be: it is not a Databricks
resource. That is the split this notebook is really about. Databricks resources
are declared and granted; everything else is a credential, and credentials go in
as secret references in Section 5.

Get the warehouse ID from your Genie space settings, or from the SQL warehouse
page, where it is the last path segment of the URL.


In [3]:
from agent import build_resources

# From your Genie space settings, or the last path segment of the warehouse URL.
WAREHOUSE_ID = "b0fffb8e3255bf85"

RESOURCES = build_resources(GENIE_SPACE_ID, WAREHOUSE_ID)
for resource in RESOURCES:
    print(type(resource).__name__, resource.to_dict())


DatabricksGenieSpace {'genie_space': [{'name': '01f1661b55731a0293c3f84ac9c5ba52'}]}
DatabricksSQLWarehouse {'sql_warehouse': [{'name': 'b0fffb8e3255bf85'}]}
DatabricksTable {'table': [{'name': 'databricks-neo4j-workshop.aircraft.aircraft'}]}
DatabricksTable {'table': [{'name': 'databricks-neo4j-workshop.aircraft.systems'}]}
DatabricksTable {'table': [{'name': 'databricks-neo4j-workshop.aircraft.sensors'}]}
DatabricksTable {'table': [{'name': 'databricks-neo4j-workshop.aircraft.sensor_readings'}]}
DatabricksTable {'table': [{'name': 'databricks-neo4j-workshop.aircraft.flights'}]}
DatabricksTable {'table': [{'name': 'databricks-neo4j-workshop.aircraft.maintenance_events'}]}
DatabricksTable {'table': [{'name': 'databricks-neo4j-workshop.aircraft.fleet_readiness'}]}
DatabricksTable {'table': [{'name': 'databricks-neo4j-workshop.aircraft.sensor_health'}]}
DatabricksServingEndpoint {'serving_endpoint': [{'name': 'databricks-meta-llama-3-3-70b-instruct'}]}
DatabricksServingEndpoint {'serving

## Section 4: Log the model

`python_model="agent.py"` is the models-from-code pattern. MLflow stores the
file rather than a pickle of an object, so what gets deployed is source you can
read, and nothing has to survive being serialized. `agent.py` ends with
`set_model(AGENT)`, which is the line that makes it the model.

`code_paths` carries `tools.py` and Lab 3's `data_utils.py` into the artifact.
Both are needed and neither is optional. `tools.py` is where the nodes and the
supervisor prompt live, and `data_utils.py` is where the embedder comes from,
the same one that wrote the vectors in your index.

**Pin `pip_requirements` rather than letting MLflow infer them.** Inference
reads the environment this notebook is running in, and your cluster carries
libraries the agent never imports. One of them, the Lab 6 memory wheel, has a
version with a local segment, `0.5.1.dev0+mentions`. A local segment resolves
from no package index, so an inferred requirement naming it produces a container
that cannot be built, and you find out fifteen minutes later in a build log.
The list below is what the agent actually imports, at the versions this workshop
installs.


In [4]:
import mlflow

PIP_REQUIREMENTS = [
    "mlflow>=3.1.0",
    "neo4j==6.2.0",
    "neo4j-graphrag>=1.17.0",
    "langgraph==1.2.4",
    "langchain-core>=1.4.6",
    "pydantic==2.13.4",
    "databricks-sdk>=0.60.0",
]

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="fleet-ops-assistant"):
    logged = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        code_paths=["tools.py", "../Lab_3_Semantic_Search/data_utils.py"],
        model_config=MODEL_CONFIG,
        resources=RESOURCES,
        pip_requirements=PIP_REQUIREMENTS,
        registered_model_name=UC_MODEL_NAME,
    )

MODEL_VERSION = logged.registered_model_version
print(f"Registered {UC_MODEL_NAME} version {MODEL_VERSION}")


2026/08/08 20:49:19 INFO mlflow.pyfunc: Predicting on input example to validate output


2026/08/08 20:49:45 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Registered model 'databricks-neo4j-workshop.agents.fleet_ops_assistant' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/10 [00:00<?, ?it/s]

Uploading artifacts:  10%|█         | 1/10 [00:00<00:05,  1.65it/s]

...

Uploading artifacts: 100%|██████████| 10/10 [00:00<00:00, 14.37it/s]

Uploading artifacts: 100%|██████████| 10/10 [00:00<00:00, 13.73it/s]

Created version '9' of model 'databricks-neo4j-workshop.agents.fleet_ops_assistant'.


🏃 View run fleet-ops-assistant at: https://dbc-cc887abc-9779.cloud.databricks.com/ml/experiments/2330184669142438/runs/75eda74b705e45669230496c9bae5ab0
🧪 View experiment at: https://dbc-cc887abc-9779.cloud.databricks.com/ml/experiments/2330184669142438


Registered databricks-neo4j-workshop.agents.fleet_ops_assistant version 9


Read back what was logged. `requirements.txt` in the artifact is the file the
serving container installs from, and it is the last cheap place to catch a
requirement that cannot resolve.


In [5]:
import pathlib

artifact = mlflow.artifacts.download_artifacts(artifact_uri=logged.model_uri)
print((pathlib.Path(artifact) / "requirements.txt").read_text())
print("Files in the model:")
for path in sorted(pathlib.Path(artifact).rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(artifact))


mlflow>=3.1.0
neo4j==6.2.0
neo4j-graphrag>=1.17.0
langgraph==1.2.4
langchain-core>=1.4.6
pydantic==2.13.4
databricks-sdk>=0.60.0
Files in the model:
  MLmodel
  __pycache__/agent.cpython-311.pyc
  agent.py
  code/data_utils.py
  code/tools.py
  conda.yaml
  input_example.json
  python_env.yaml
  registered_model_meta
  requirements.txt
  serving_input_example.json


## Section 5: Credentials, which are not resources

Your Aura password cannot be declared, granted, or looked up. It has to travel.

It travels as a **secret reference**. `{{secrets/<scope>/<key>}}` is a string
Model Serving resolves when it applies the endpoint configuration, so what is
written into the endpoint is the reference and what exists in the container is
the value. The password is never in this notebook, never in MLflow, and never in
the endpoint's own configuration.

`serving_environment_vars` builds all three from your scope name. Print the keys
if you like. The values are references rather than secrets, which is why they
are safe to look at.


In [6]:
from agent import serving_environment_vars

ENVIRONMENT_VARS = serving_environment_vars(SECRET_SCOPE)
for name, reference in ENVIRONMENT_VARS.items():
    print(f"{name} = {reference}")


NEO4J_URI = {{secrets/fleet-ops-ryan-knight-neo4j-com/neo4j-uri}}
NEO4J_USERNAME = {{secrets/fleet-ops-ryan-knight-neo4j-com/neo4j-username}}
NEO4J_PASSWORD = {{secrets/fleet-ops-ryan-knight-neo4j-com/neo4j-password}}
NEO4J_DATABASE = {{secrets/fleet-ops-ryan-knight-neo4j-com/neo4j-database}}


### Deploy

`agents.deploy` creates the endpoint, attaches the model version, applies the
environment block, and grants the serving principal the resources you declared.

Setting an active MLflow experiment first is what makes the deployed agent log
its traces back to MLflow. Without it the endpoint still works and you get no
traces, which is a poor trade when the thing you most want to see is which tool
ran.

This returns in under a minute and the endpoint is not ready when it does.


In [7]:
from databricks import agents

mlflow.set_experiment(f"/Users/{CURRENT_USER}/{ENDPOINT_NAME}")

deployment = agents.deploy(
    UC_MODEL_NAME,
    MODEL_VERSION,
    endpoint_name=ENDPOINT_NAME,
    environment_vars=ENVIRONMENT_VARS,
    scale_to_zero=True,
)
print("Endpoint:", deployment.endpoint_name)
print("Query URL:", deployment.query_endpoint)


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


/private/tmp/claude-502/-Users-ryanknight-projects-databricks-databricks-neo4j-workshop/ac6ac594-016c-4399-b151-dd5a9b39b221/scratchpad/lab5venv/lib/python3.11/site-packages/databricks/agents/deployments.py:641: UserWarning: This endpoint is being deployed without a feedback model, which has been deprecated.
For more information, see: https://docs.databricks.com/aws/en/generative-ai/agent-framework/feedback-model
  warnings.warn(



    Deployment of databricks-neo4j-workshop.agents.fleet_ops_assistant version 9 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://dbc-cc887abc-9779.cloud.databricks.com/ml/endpoints/fleet-ops-assistant-ryan-knight-neo4j-com/
    Review App: https://dbc-cc887abc-9779.cloud.databricks.com/ml/review-v2/0622b05cb4de4fb78e3fdd01f7c0a768/chat

You can refer back to the links above from the endpoint detail page at https://dbc-cc887abc-9779.cloud.databricks.com/ml/endpoints/fleet-ops-assistant-ryan-knight-neo4j-com/.

To set up monitoring for your deployed agent, see:
https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/production-monitoring
Endpoint: fleet-ops-assistant-ryan-knight-neo4j-com
Query URL: https://dbc-cc887abc-9779.cloud.databricks.com/serving-endpoints/fleet-ops-assistant-ryan-knight-neo4j-com/served-models/databricks-neo4j-workshop-agents-fleet_ops_assistant_9/invocat

## Section 6: Wait for it

A first deploy builds a container from the requirements you pinned, which takes
roughly ten to fifteen minutes. The cell below polls until the endpoint is ready
and prints where it got to, so you can watch instead of guessing.


In [8]:
import time

from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()
started = time.time()

while True:
    endpoint = workspace.serving_endpoints.get(ENDPOINT_NAME)
    ready = endpoint.state.ready.value
    updating = endpoint.state.config_update.value
    print(f"[{time.time() - started:6.0f}s] ready={ready} config_update={updating}")
    if updating == "UPDATE_FAILED":
        raise RuntimeError("Deployment failed. Open the endpoint page and read the build logs.")
    if ready == "READY" and updating == "NOT_UPDATING":
        break
    if time.time() - started > 1800:
        raise TimeoutError("Endpoint did not become ready within 30 minutes.")
    time.sleep(30)

print(f"\nReady after {(time.time() - started) / 60:.1f} minutes.")


[     1s] ready=READY config_update=IN_PROGRESS


[    31s] ready=READY config_update=IN_PROGRESS


[    61s] ready=READY config_update=IN_PROGRESS


[    91s] ready=READY config_update=IN_PROGRESS


[   121s] ready=READY config_update=IN_PROGRESS


...

[   423s] ready=READY config_update=IN_PROGRESS


[   454s] ready=READY config_update=IN_PROGRESS


[   484s] ready=READY config_update=IN_PROGRESS


[   514s] ready=READY config_update=IN_PROGRESS


[   544s] ready=READY config_update=NOT_UPDATING

Ready after 9.1 minutes.


## Section 7: Ask the endpoint

The endpoint speaks the Responses API, so a question is one user message and the
answer comes back as output items. `custom_outputs` carries the route, which is
the part worth reading first: an answer with a strange route is a different bug
from an answer with a sensible one.


In [9]:
from mlflow.deployments import get_deploy_client

deploy_client = get_deploy_client("databricks")


def ask_endpoint(question: str) -> dict:
    """Send one question to the endpoint and unpack the reply."""
    response = deploy_client.predict(
        endpoint=ENDPOINT_NAME,
        inputs={"input": [{"role": "user", "content": question}]},
    )
    text = "".join(
        part.get("text", "")
        for item in response.get("output", [])
        for part in item.get("content", [])
    )
    custom = response.get("custom_outputs") or {}
    return {
        "answer": text,
        "trace": custom.get("trace", []),
        "usage": response.get("usage") or {},
    }


result = ask_endpoint("What is the procedure for an EGT exceedance?")
print("Route:", result["trace"])
print("Usage:", result["usage"])
print(result["answer"][:800])


Route: ['graphrag_node']
Usage: {}
The procedure for an EGT exceedance is outlined in the troubleshooting procedure section and includes steps such as reviewing DFDR/QAR data for "EGT exceedance duration" and documenting "peak temperature" and "exposure time", performing a visual inspection of the exhaust nozzle to check for "discoloration or distortion", conducting a borescope inspection of the HPT blades to check for "thermal distress, coating loss, or other damage", verifying EGT probe calibration by comparing "dual probe readings" and checking for any "variance", and checking the fuel nozzle spray pattern to ensure "even fuel distribution". The severity of the EGT exceedance determines the required actions, which are categorized as CRITICAL, MAJOR, or MINOR, with specific requirements such as grounding the aircraft for 


One question per tool, through the endpoint rather than through the graph in
this notebook. Same three tools, same routing prompt, none of your credentials.

The check is whether the tool that could answer was called, not whether it was
the only one. A tool that comes back empty sends the supervisor round again, so
a route of two or three names on a single-tool question is the retry working
rather than the routing failing.


In [10]:
PER_TOOL_QUESTIONS = [
    ("genie_node", "What is the average EGT for aircraft N10000?"),
    ("cypher_node", "Which components are in the hydraulic system of N10000?"),
    ("graphrag_node", "What is the procedure for an EGT exceedance?"),
]

for expected, question in PER_TOOL_QUESTIONS:
    result = ask_endpoint(question)
    called = result["trace"]
    verdict = "OK " if expected in called else "HUH"
    print(f"{verdict} {expected:14} -> {called}")
    print(f"     {question}")


OK  genie_node     -> ['genie_node']
     What is the average EGT for aircraft N10000?


OK  cypher_node    -> ['cypher_node']
     Which components are in the hydraulic system of N10000?


OK  graphrag_node  -> ['graphrag_node']
     What is the procedure for an EGT exceedance?


### The anchor question

The one that needs all three. Sensor readings live only in Delta, maintenance
history lives only in the graph, and the procedure lives only in the manuals, so
no single tool can finish it.


In [11]:
ANCHOR = (
    "Which engines are showing abnormal EGT readings, what maintenance history "
    "do those aircraft have, and what does the maintenance manual say to do "
    "about high EGT?"
)

anchor_result = ask_endpoint(ANCHOR)
print("Route:", anchor_result["trace"])
print()
print(anchor_result["answer"])


Route: ['genie_node', 'cypher_node', 'graphrag_node']

A total of 71 engines across multiple aircraft are showing abnormal EGT readings, with most having 540 such events, indicating a widespread and persistent issue. Notable data points include **N10034, CFM56-7B #1** and **N10034, CFM56-7B #2** with 540 abnormal readings, **N10007, LEAP-1A #2** with 540 abnormal readings, **N10011, CFM56-7B #2** with 540 abnormal readings, and **N10019, LEAP-1A #1** with 540 abnormal readings. The maintenance manual recommends corrective actions for high EGT readings, including **CRITICAL** actions such as engine removal for shop cleaning with an estimated downtime of "3-5 days", **MAJOR** actions like performing on-wing compressor wash, and **MINOR** actions like scheduling compressor wash at the next opportunity. The manual also provides guidance on detecting contamination issues, such as reviewing trend data for EGT margin degradation, performing oil spectrographic analysis, checking fuel filter di

## Section 8: Evaluate it

Two things are worth scoring and they are not the same thing.

**Routing** is deterministic. Either the supervisor called the tool that can
answer or it did not, and no judge is needed to tell you which. That is a plain
Python function, decorated as a scorer.

**The answer** is not deterministic, so it gets an LLM judge. `Correctness`
compares the answer against what you said a good answer contains.
`RelevanceToQuery` catches the answer that is true and about something else.

Both run over the same evaluation set, and `predict_fn` is the endpoint, so what
is being scored is the deployed model rather than a copy of it running here.


In [12]:
EVAL_PAIRS = [
    {
        "inputs": {"question": "What is the average EGT for aircraft N10000?"},
        "expectations": {
            "expected_tools": ["genie_node"],
            "expected_facts": ["an average EGT value in degrees Celsius for N10000"],
        },
    },
    {
        "inputs": {
            "question": "Which components are in the hydraulic system of N10000?"
        },
        "expectations": {
            "expected_tools": ["cypher_node"],
            "expected_facts": ["a pump", "a filter", "a reservoir", "an actuator"],
        },
    },
    {
        "inputs": {"question": "What is the procedure for an EGT exceedance?"},
        "expectations": {
            "expected_tools": ["graphrag_node"],
            "expected_facts": [
                "borescope inspection of the high pressure turbine",
                "verifying EGT probe calibration",
            ],
        },
    },
    {
        "inputs": {
            "question": "What maintenance events has aircraft N10004 had, and how severe were they?"
        },
        "expectations": {
            "expected_tools": ["cypher_node"],
            "expected_facts": ["maintenance events for N10004 with a severity"],
        },
    },
    {
        "inputs": {"question": "How do I troubleshoot engine vibration?"},
        "expectations": {
            "expected_tools": ["graphrag_node"],
            "expected_facts": ["a vibration troubleshooting step from the manual"],
        },
    },
    {
        "inputs": {"question": ANCHOR},
        "expectations": {
            "expected_tools": ["genie_node", "cypher_node", "graphrag_node"],
            "expected_facts": [
                "engines with elevated EGT",
                "maintenance history for those aircraft",
                "what the manual says to do about high EGT",
            ],
        },
    },
]
print(f"{len(EVAL_PAIRS)} evaluation pairs")


6 evaluation pairs


In [13]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, scorer


@scorer
def routing(outputs: dict, expectations: dict) -> float:
    """Fraction of the expected tools the supervisor actually called.

    Deterministic on purpose. A judge asked whether the right tool ran would be
    guessing at something the trace already states.
    """
    expected = set(expectations.get("expected_tools", []))
    if not expected:
        return 1.0
    called = set(outputs.get("trace", []))
    return len(expected & called) / len(expected)


def predict_fn(question: str) -> dict:
    """Send one evaluation question to the deployed endpoint.

    Returns the answer and the route together, because the two scorers need
    different halves of it and MLflow hands both the same `outputs`.
    """
    result = ask_endpoint(question)
    return {"answer": result["answer"], "trace": result["trace"]}


results = mlflow.genai.evaluate(
    data=EVAL_PAIRS,
    predict_fn=predict_fn,
    scorers=[routing, Correctness(), RelevanceToQuery()],
)
results.tables["eval_results"]


2026/08/08 21:01:57 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.


2026/08/08 21:01:57 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/6 [Elapsed: 00:00, Remaining: ?]

Evaluating:  17%|█▋        | 1/6 [Elapsed: 00:31, Remaining: 02:35]

Evaluating:  33%|███▎      | 2/6 [Elapsed: 00:31, Remaining: 01:02]

...

Evaluating:  83%|████████▎ | 5/6 [Elapsed: 00:36, Remaining: 00:07]

Evaluating: 100%|██████████| 6/6 [Elapsed: 00:54, Remaining: 00:00]

Evaluating: 100%|██████████| 6/6 [Elapsed: 00:54, Remaining: 00:00] [predict_fn: 68%, scorers: 32%]

Evaluating: 100%|██████████| 6/6 [Elapsed: 00:54, Remaining: 00:00] [predict_fn: 68%, scorers: 32%]

,trace_id,expected_facts/value,relevance_to_query/value,relevance_to_query/rationale,routing/value,correctness/value,correctness/rationale,expected_tools/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-2c8ba84f43a58f9bc7a86453d57c3ffb,None,yes,The question asks specifically for the average...,1.0,yes,The response provides a specific average EGT v...,None,"{""info"": {""trace_id"": ""tr-2c8ba84f43a58f9bc7a8...",tr-2c8ba84f43a58f9bc7a86453d57c3ffb,OK,1786244543425,16812,{'question': 'What is the average EGT for airc...,{'answer': 'The average EGT for aircraft N1000...,"{'mlflow.trace.sizeBytes': '3038', 'mlflow.tra...",{'mlflow.eval.requestId': '0d1cddf2-e5f6-42f5-...,"[{'trace_id': 'LIuoT0Olj5vHqGRT1Xw/+w==', 'spa...",[{'assessment_id': 'a-fa88a45b9ab847d6a5f43417...
1,tr-633e68247f66964957c0d26123733b0a,None,yes,The question asks which components are in the ...,1.0,yes,The response states that the hydraulic system ...,None,"{""info"": {""trace_id"": ""tr-633e68247f66964957c0...",tr-633e68247f66964957c0d26123733b0a,OK,1786244543426,11543,{'question': 'Which components are in the hydr...,{'answer': 'The hydraulic system of N10000 con...,"{'mlflow.trace.sizeBytes': '3119', 'mlflow.tra...",{'mlflow.eval.requestId': '254e42fb-1272-41d0-...,"[{'trace_id': 'Yz5oJH9mlklXwNJhI3M7Cg==', 'spa...",[{'assessment_id': 'a-085b1cfd919d4108bcc26c15...
2,tr-94dcae6775abb64b5514aaf7a685e8b7,None,yes,The question asks for the procedure for an EGT...,1.0,yes,The claim mentions two specific actions: 'bore...,None,"{""info"": {""trace_id"": ""tr-94dcae6775abb64b5514...",tr-94dcae6775abb64b5514aaf7a685e8b7,OK,1786244543426,12753,{'question': 'What is the procedure for an EGT...,{'answer': 'The procedure for an EGT exceedanc...,"{'mlflow.trace.sizeBytes': '4938', 'mlflow.tra...",{'mlflow.eval.requestId': 'f0b17719-185d-4456-...,"[{'trace_id': 'lNyuZ3WrtktVFKr3poXotw==', 'spa...",[{'assessment_id': 'a-6a86d215889541138882a68e...
3,tr-78a66d0297f2b99ec8e4cce8551d6ded,None,yes,The question asks about the maintenance events...,1.0,yes,The response provides detailed information abo...,None,"{""info"": {""trace_id"": ""tr-78a66d0297f2b99ec8e4...",tr-78a66d0297f2b99ec8e4cce8551d6ded,OK,1786244543427,7818,{'question': 'What maintenance events has airc...,{'answer': 'Aircraft N10004 had 23 maintenance...,"{'mlflow.trace.sizeBytes': '4221', 'mlflow.tra...",{'mlflow.eval.requestId': '086f6bd2-9fa7-4b6e-...,"[{'trace_id': 'eKZtApfyuZ7I5MzoVR1t7Q==', 'spa...",[{'assessment_id': 'a-7182654d550f43218b257984...
4,tr-c579ecd3bbed2804df959524b156c604,None,yes,The answer provides detailed steps on how to t...,1.0,yes,The response provides a detailed troubleshooti...,None,"{""info"": {""trace_id"": ""tr-c579ecd3bbed2804df95...",tr-c579ecd3bbed2804df959524b156c604,OK,1786244543426,12552,{'question': 'How do I troubleshoot engine vib...,"{'answer': 'To troubleshoot engine vibration, ...","{'mlflow.trace.sizeBytes': '4079', 'mlflow.tra...",{'mlflow.eval.requestId': '9efb1df2-efe3-49b9-...,"[{'trace_id': 'xXns07vtKATflZUksVbGBA==', 'spa...",[{'assessment_id': 'a-03bf61af00444299bda1c6eb...
5,tr-d18d37da7a85f66e4f7b8c472c81ce61,None,no,The answer discusses the engines showing abnor...,1.0,no,The response states that the engines showing a...,None,"{""info"": {""trace_id"": ""tr-d18d37da7a85f66e4f7b...",tr-d18d37da7a85f66e4f7b8c472c81ce61,OK,1786244543427,33383,{'question': 'Which engines are showing abnorm...,{'answer': 'The engines showing abnormal EGT r...,"{'mlflow.trace.sizeBytes': '5178', 'mlflow.tra...",{'mlflow.eval.requestId': 'a43bdcbc-3a7c-4472-...,"[{'trace_id': '0Y032nqF9m5Pe4xHLIHOYQ==', 'spa...",[{'assessment_id': 'a-49a371ef6e7d45ff9ed7d34c...


Open the run in the MLflow experiment to read the traces beside the scores. A
low `Correctness` with `routing` at 1.0 means the right tool ran and answered
badly, which is a prompt or a data problem. A low `routing` means the supervisor
never asked the tool that had the answer, which is Section 6 of notebook 01.


## Section 9: When it fails

Three failures account for nearly all of them, and each has one thing to check.

**The endpoint answers, but says it could not open its Neo4j connection.** The
model loaded and the credentials did not arrive. Check that the endpoint carries
all three environment variables, on the endpoint page under the served entity,
and that the scope named in them is the scope you created in Lab 3. The agent
prints the missing variable by name, so read the message before changing
anything.

**`genie_node` says it is not authorized to use or monitor this SQL Endpoint.**
The `DatabricksSQLWarehouse` resource is missing or names the wrong warehouse.
Check `WAREHOUSE_ID` in Section 3 against your Genie space settings, then log
and deploy again. Nothing else has to change, and the routing you already
verified will be unaffected.

**The deploy never reaches READY.** The container could not be built, and the
reason is in the build logs on the endpoint page rather than in this notebook. A
requirement that cannot resolve is the usual cause, which is why Section 4 pins
them and prints the file back.

An endpoint deployed with `scale_to_zero=True` sleeps when it is idle, so the
first question after a quiet period takes longer while it wakes. That is not a
failure.


## What you built

The notebook 01 agent, running somewhere else, as something other than you.

The three ideas worth taking away are the split between resources and
credentials, the fact that a served model is source rather than a pickle, and
scoring routing separately from answers, because the two fail for different
reasons and the fix lives in different files.

Lab 6 redeploys this same endpoint with memory in Neo4j, so it can be asked a
follow-up.
